# Report walkthrough

This is the interactive counterpart to a report's `report_summary.md`. It reads an already-generated `reports/<venue>/<composite>/<run-id>/` directory -- written by either `run_composite` or the standalone `simcast.cli.report_composite` -- and displays the same tables and figures interactively. It does not fit, evaluate, or write anything unless the optional regeneration cell near the end is explicitly enabled.

See [the usage guide](../../docs/technical/usage_guide.md#15-regenerating-or-customizing-a-report), [the report configuration reference](../../docs/technical/configuration_reference.md#7-report-configuration), and [the artifact reference](../../docs/technical/artifact_reference.md#composite-experiment-and-report-roots).

In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

REPORT_DIR = '/home/kbolat/Python/simcast/reports/lab/quick_all_methods/2026-09-22_085051'
if REPORT_DIR is None:
    candidates = sorted((PROJECT_ROOT / 'reports').rglob('report_summary.md'))
    REPORT_DIR = candidates[-1].parent if candidates else None
else:
    REPORT_DIR = Path(REPORT_DIR).expanduser()
    if not REPORT_DIR.is_absolute():
        REPORT_DIR = PROJECT_ROOT / REPORT_DIR
    if (REPORT_DIR / 'composite_manifest.json').is_file():
        run_parts = REPORT_DIR.parts
        if 'runs' not in run_parts:
            raise ValueError(f'Cannot derive report path from run directory: {REPORT_DIR}')
        run_index = run_parts.index('runs')
        REPORT_DIR = Path(*run_parts[:run_index], 'reports', *run_parts[run_index + 1:])

if REPORT_DIR is not None:
    REPORT_DIR = REPORT_DIR.resolve()
if REPORT_DIR is None or not (REPORT_DIR / 'report_summary.md').is_file():
    raise FileNotFoundError(f'Report summary not found under {REPORT_DIR}')
print(REPORT_DIR)

/home/kbolat/Python/simcast/reports/lab/quick_all_methods/2026-09-22_085051


## Report summary

Every report directory contains a generated `report_summary.md` naming its metrics, reference, and bootstrap settings, and describing every accompanying file. It is the same plain-text index whether the report came from `run_composite` or `report_composite`.

In [ ]:
from IPython.display import Markdown, display

if REPORT_DIR is not None:
    display(Markdown((REPORT_DIR / 'report_summary.md').read_text(encoding='utf-8')))

# Report summary

Metrics: mean_pinball.
Reference: `m0`. Bootstrap: 200 replicates, block lengths [7, 3].

| File | Contents |
|---|---|
| `method_comparison_mean_pinball.pdf` | bar chart of `mean_pinball` per experiment/method (pdf) |
| `method_comparison_mean_pinball.png` | bar chart of `mean_pinball` per experiment/method (png) |
| `method_summary.csv` | mean value per (base, experiment, method, seed), one row per metric |
| `paired_effects.csv` | moving-block bootstrap paired contrasts vs. the reference, one row per metric/comparison |
| `per_origin_metrics.parquet` | concatenated per-origin, per-method evaluation records; input to every file below |


## Method summary and paired effects

`method_summary.csv` is the mean value of each configured metric per base, experiment, method, and seed. `paired_effects.csv` is the moving-block bootstrap contrast of each method against the declared reference, one row per metric, comparison, and block length. Both are read here exactly as written; nothing is recomputed.

In [ ]:
import pandas as pd

if REPORT_DIR is not None:
    method_summary = pd.read_csv(REPORT_DIR / 'method_summary.csv')
    paired_effects = pd.read_csv(REPORT_DIR / 'paired_effects.csv')
    display(method_summary)
    display(paired_effects)

## Method comparison figures

One `method_comparison_<metric>.png`/`.pdf` pair exists per configured metric. They are shown inline here exactly as saved; nothing is replotted.

In [ ]:
from IPython.display import Image

if REPORT_DIR is not None:
    for path in sorted(REPORT_DIR.glob('method_comparison_*.png')):
        print(path.name)
        display(Image(filename=str(path)))

## Regenerating this report with different metrics or analyses

`report_composite` reads only the `composite_manifest.json` already sitting under `run_root`; it never calls Chronos, a dependence fitter, or the evaluator. This makes it safe to rerun repeatedly while iterating on which metrics, bootstrap settings, or figures to produce. Disabled by default so opening this notebook never writes anything.

In [ ]:
from simcast.cli.report_composite import report_composite

REGENERATE = False  # set True to actually write a new report directory
REPORT_CONFIG = 'configs/reports/powertech2027_main.yaml'
if REGENERATE:
    new_report_dir = report_composite(PROJECT_ROOT / REPORT_CONFIG)
    print(new_report_dir)

## Configuration and implementation map

`metrics` and `analysis` in a `report` YAML document (Section 7 of the [configuration reference](../../docs/technical/configuration_reference.md#7-report-configuration)) control what this notebook displays. `simcast.reporting.composite_report.write_composite_report` builds every file above; `simcast.cli.report_composite` is the CLI entry point used in the regeneration cell. The [artifact reference](../../docs/technical/artifact_reference.md#composite-experiment-and-report-roots) defines the exact schema of each file.